In [1]:
from pathlib import Path
import pandas as pd
import duckdb

# Project root
project_dir = Path("E:/Cloud_cost_capstone")

# Create a folder for our database
db_dir = project_dir / "data" / "database"
db_dir.mkdir(parents=True, exist_ok=True)

# DuckDB database location
db_path = db_dir / "cloud_workload.duckdb"

# Establish connection
con = duckdb.connect(str(db_path))

print("DuckDB connection successful!")
print("Database:", db_path)

DuckDB connection successful!
Database: E:\Cloud_cost_capstone\data\database\cloud_workload.duckdb


In [2]:
result = con.execute("""
    SELECT
        'Bitbrains' AS dataset,
        100 AS selected_vms,
        CURRENT_TIMESTAMP AS query_time
""").fetchdf()

display(result)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19076\2412557585.py:6: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  """).fetchdf()


,dataset,selected_vms,query_time
0,Bitbrains,100,2026-11-17 16:47:18.058615+01:00


In [3]:
# Locate the dataset and saved VM selection
data_dir = project_dir / "data/raw/fastStorage/2013-8"
selection_file = project_dir / "reports/selected_vm_ids.csv"

# Load saved VM identifiers
selected_df = pd.read_csv(selection_file)

selected_ids = (
    selected_df.iloc[:, 0]
    .astype(str)
    .str.strip()
    .map(lambda x: str(int(float(x))) if x.replace(".", "", 1).isdigit()
         else Path(x).stem)
)

# Match VM identifiers to original CSV files
file_lookup = {
    file.stem: file
    for file in data_dir.rglob("*.csv")
}

selected_files = [
    file_lookup[vm_id]
    for vm_id in selected_ids
]

print("Selected files:", len(selected_files))
print("Unique files:", len(set(selected_files)))
print("All files exist:", all(f.exists() for f in selected_files))

Selected files: 100
Unique files: 100
All files exist: True


In [4]:
con.execute("""
CREATE TABLE IF NOT EXISTS vm_workload_raw (
    vm_id VARCHAR,
    timestamp_raw BIGINT,
    cpu_usage_pct DOUBLE,
    cpu_usage_mhz DOUBLE,
    cpu_capacity_mhz DOUBLE,
    memory_usage_kb DOUBLE,
    memory_capacity_kb DOUBLE,
    disk_read_kbps DOUBLE,
    disk_write_kbps DOUBLE,
    network_received_kbps DOUBLE,
    network_transmitted_kbps DOUBLE
)
""")

print("Staging table created successfully!")

Staging table created successfully!


In [6]:
# Clear table to prevent duplicate ingestion on reruns
con.execute("DELETE FROM vm_workload_raw")

for file in selected_files:

    vm = pd.read_csv(file, sep=";")
    vm.columns = vm.columns.str.strip()

    vm_clean = pd.DataFrame({
        "vm_id": file.stem,
        "timestamp_raw": vm["Timestamp [ms]"],
        "cpu_usage_pct": vm["CPU usage [%]"],
        "cpu_usage_mhz": vm["CPU usage [MHZ]"],
        "cpu_capacity_mhz": vm["CPU capacity provisioned [MHZ]"],
        "memory_usage_kb": vm["Memory usage [KB]"],
        "memory_capacity_kb": vm["Memory capacity provisioned [KB]"],
        "disk_read_kbps": vm["Disk read throughput [KB/s]"],
        "disk_write_kbps": vm["Disk write throughput [KB/s]"],
        "network_received_kbps": vm["Network received throughput [KB/s]"],
        "network_transmitted_kbps": vm["Network transmitted throughput [KB/s]"]
    })

    con.register("vm_batch", vm_clean)

    con.execute("""
        INSERT INTO vm_workload_raw
        SELECT * FROM vm_batch
    """)

    con.unregister("vm_batch")

print("Ingestion completed!")

Ingestion completed!


In [7]:
result = con.execute("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT vm_id) AS total_vms,
    MIN(timestamp_raw) AS earliest_timestamp,
    MAX(timestamp_raw) AS latest_timestamp
FROM vm_workload_raw
""").fetchdf()

display(result)

,total_records,total_vms,earliest_timestamp,latest_timestamp
0,890306,100,1376314846,1378906798


In [8]:
con.execute("""
CREATE OR REPLACE TABLE vm_workload_prepared AS
SELECT
    *,
    to_timestamp(timestamp_raw) AS datetime_utc,

    CASE
        WHEN cpu_capacity_mhz > 0
        THEN cpu_usage_mhz / cpu_capacity_mhz * 100
        ELSE NULL
    END AS calculated_cpu_pct,

    CASE
        WHEN memory_capacity_kb > 0
        THEN memory_usage_kb / memory_capacity_kb * 100
        ELSE NULL
    END AS memory_util_pct,

    (cpu_capacity_mhz <= 0) AS invalid_cpu_capacity,
    (memory_capacity_kb <= 0) AS invalid_memory_capacity,
    (cpu_usage_pct < 0 OR cpu_usage_pct > 100)
        AS cpu_pct_out_of_range,
    (memory_usage_kb = 0) AS zero_memory_flag

FROM vm_workload_raw
""")

print("Prepared workload table created!")

Prepared workload table created!


In [9]:
result = con.execute("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT vm_id) AS total_vms,
    SUM(CAST(invalid_cpu_capacity AS INTEGER))
        AS invalid_cpu_records,
    SUM(CAST(invalid_memory_capacity AS INTEGER))
        AS invalid_memory_records
FROM vm_workload_prepared
""").fetchdf()

display(result)

,total_records,total_vms,invalid_cpu_records,invalid_memory_records
0,890306,100,105091.0,105072.0


In [10]:
con.execute("""
CREATE OR REPLACE TABLE vm_workload_dedup AS

WITH distinct_records AS (
    -- Remove exact duplicate records
    SELECT DISTINCT *
    FROM vm_workload_prepared
),

ranked AS (
    SELECT
        *,
        COUNT(*) OVER (
            PARTITION BY vm_id, timestamp_raw
        ) AS records_at_timestamp,

        ROW_NUMBER() OVER (
            PARTITION BY vm_id, timestamp_raw
            ORDER BY
                cpu_usage_pct NULLS LAST,
                cpu_usage_mhz NULLS LAST,
                cpu_capacity_mhz NULLS LAST,
                memory_usage_kb NULLS LAST,
                memory_capacity_kb NULLS LAST,
                disk_read_kbps NULLS LAST,
                disk_write_kbps NULLS LAST,
                network_received_kbps NULLS LAST,
                network_transmitted_kbps NULLS LAST
        ) AS rn

    FROM distinct_records
)

SELECT
    * EXCLUDE (rn),
    (records_at_timestamp > 1)
        AS conflicting_timestamp_flag

FROM ranked
WHERE rn = 1
""")

print("Deduplicated table created successfully!")

Deduplicated table created successfully!


In [11]:
result = con.execute("""
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT vm_id) AS total_vms,

    COUNT(*) - COUNT(DISTINCT (vm_id, timestamp_raw))
        AS remaining_duplicate_timestamps,

    COUNT(*) FILTER (
        WHERE conflicting_timestamp_flag
    ) AS conflicting_timestamps

FROM vm_workload_dedup
""").fetchdf()

display(result)

,total_records,total_vms,remaining_duplicate_timestamps,conflicting_timestamps
0,857977,100,0,40


In [12]:
con.execute("""
CREATE OR REPLACE TABLE vm_workload_hourly AS

SELECT
    vm_id,
    DATE_TRUNC('hour', datetime_utc) AS hour_utc,

    COUNT(*) AS sample_count,

    -- CPU
    AVG(cpu_usage_pct) FILTER (
        WHERE NOT invalid_cpu_capacity
          AND NOT cpu_pct_out_of_range
          AND NOT conflicting_timestamp_flag
    ) AS cpu_avg_pct,

    MAX(cpu_usage_pct) FILTER (
        WHERE NOT invalid_cpu_capacity
          AND NOT cpu_pct_out_of_range
          AND NOT conflicting_timestamp_flag
    ) AS cpu_max_pct,

    -- Memory
    AVG(memory_util_pct) FILTER (
        WHERE NOT invalid_memory_capacity
          AND memory_util_pct BETWEEN 0 AND 100
          AND NOT conflicting_timestamp_flag
    ) AS memory_avg_pct,

    -- Disk
    AVG(disk_read_kbps) AS disk_read_avg_kbps,
    AVG(disk_write_kbps) AS disk_write_avg_kbps,

    -- Network
    AVG(network_received_kbps) AS network_received_avg_kbps,
    AVG(network_transmitted_kbps) AS network_transmitted_avg_kbps,

    -- Quality indicators
    COUNT(*) FILTER (
        WHERE NOT invalid_cpu_capacity
          AND NOT cpu_pct_out_of_range
          AND NOT conflicting_timestamp_flag
    ) AS valid_cpu_samples,

    COUNT(*) FILTER (
        WHERE NOT invalid_memory_capacity
          AND memory_util_pct BETWEEN 0 AND 100
          AND NOT conflicting_timestamp_flag
    ) AS valid_memory_samples,

    COUNT(*) FILTER (
        WHERE conflicting_timestamp_flag
    ) AS conflicting_samples

FROM vm_workload_dedup

GROUP BY
    vm_id,
    DATE_TRUNC('hour', datetime_utc)

ORDER BY vm_id, hour_utc
""")

print("Hourly workload table created successfully!")

Hourly workload table created successfully!


In [13]:
result = con.execute("""
SELECT
    COUNT(*) AS total_hourly_records,
    COUNT(DISTINCT vm_id) AS total_vms,
    COUNT(*) - COUNT(DISTINCT (vm_id, hour_utc))
        AS duplicate_vm_hours,
    MIN(sample_count) AS min_samples_per_hour,
    MAX(sample_count) AS max_samples_per_hour
FROM vm_workload_hourly
""").fetchdf()

display(result)

,total_hourly_records,total_vms,duplicate_vm_hours,min_samples_per_hour,max_samples_per_hour
0,68852,100,0,2,24


In [14]:
con.execute("""
CREATE OR REPLACE TABLE vm_hourly_cost AS

SELECT
    *,

    CASE
        WHEN valid_cpu_samples >= 9
         AND valid_memory_samples >= 9
         AND cpu_avg_pct BETWEEN 0 AND 100
         AND memory_avg_pct BETWEEN 0 AND 100

        THEN
            0.5 * (cpu_avg_pct / 100.0)
          + 0.5 * (memory_avg_pct / 100.0)

        ELSE NULL
    END AS resource_cost_index,

    CASE
        WHEN valid_cpu_samples >= 9
         AND valid_memory_samples >= 9
         AND cpu_avg_pct BETWEEN 0 AND 100
         AND memory_avg_pct BETWEEN 0 AND 100
        THEN TRUE
        ELSE FALSE
    END AS valid_cost_hour

FROM vm_workload_hourly
""")

print("Hourly resource-cost index created successfully!")

Hourly resource-cost index created successfully!


In [15]:
result = con.execute("""
SELECT
    COUNT(*) AS total_hours,

    COUNT(resource_cost_index) AS valid_cost_hours,

    COUNT(*) - COUNT(resource_cost_index)
        AS excluded_cost_hours,

    ROUND(MIN(resource_cost_index), 4) AS min_cost_index,

    ROUND(AVG(resource_cost_index), 4) AS avg_cost_index,

    ROUND(MAX(resource_cost_index), 4) AS max_cost_index

FROM vm_hourly_cost
""").fetchdf()

display(result)

,total_hours,valid_cost_hours,excluded_cost_hours,min_cost_index,avg_cost_index,max_cost_index
0,68852,63125,5727,0.0,0.0728,0.8795


In [16]:
con.execute("""
CREATE OR REPLACE TABLE vm_forecast_base AS
SELECT
    current_vm.*,

    next_vm.resource_cost_index AS target_next_hour

FROM vm_hourly_cost AS current_vm

LEFT JOIN vm_hourly_cost AS next_vm
    ON current_vm.vm_id = next_vm.vm_id
    AND next_vm.hour_utc =
        current_vm.hour_utc + INTERVAL '1 hour'
""")

print("Next-hour forecasting target created!")

Next-hour forecasting target created!


In [17]:
result = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(target_next_hour) AS available_targets,
    COUNT(*) - COUNT(target_next_hour) AS unavailable_targets
FROM vm_forecast_base
""").fetchdf()

display(result)

,total_rows,available_targets,unavailable_targets
0,68852,63125,5727


In [18]:
con.execute("""
CREATE OR REPLACE TABLE vm_forecast_features AS

SELECT
    current_vm.*,

    -- Historical cost index
    lag1.resource_cost_index AS cost_lag_1h,
    lag2.resource_cost_index AS cost_lag_2h,

    -- Historical workload features
    lag1.cpu_avg_pct AS cpu_lag_1h,
    lag1.memory_avg_pct AS memory_lag_1h,
    lag1.disk_read_avg_kbps AS disk_read_lag_1h,
    lag1.disk_write_avg_kbps AS disk_write_lag_1h,
    lag1.network_received_avg_kbps AS network_received_lag_1h,
    lag1.network_transmitted_avg_kbps AS network_transmitted_lag_1h,

    -- Recent 3-hour averages, including current hour
    (
        current_vm.cpu_avg_pct
        + lag1.cpu_avg_pct
        + lag2.cpu_avg_pct
    ) / 3.0 AS cpu_rolling_3h,

    (
        current_vm.memory_avg_pct
        + lag1.memory_avg_pct
        + lag2.memory_avg_pct
    ) / 3.0 AS memory_rolling_3h,

    (
        current_vm.resource_cost_index
        + lag1.resource_cost_index
        + lag2.resource_cost_index
    ) / 3.0 AS cost_rolling_3h

FROM vm_forecast_base AS current_vm

LEFT JOIN vm_hourly_cost AS lag1
    ON current_vm.vm_id = lag1.vm_id
    AND lag1.hour_utc =
        current_vm.hour_utc - INTERVAL '1 hour'

LEFT JOIN vm_hourly_cost AS lag2
    ON current_vm.vm_id = lag2.vm_id
    AND lag2.hour_utc =
        current_vm.hour_utc - INTERVAL '2 hours'
""")

print("Historical forecasting features created!")

Historical forecasting features created!


In [19]:
result = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT vm_id) AS total_vms,
    COUNT(target_next_hour) AS available_targets,
    COUNT(cost_lag_1h) AS available_cost_lag_1h,
    COUNT(cost_rolling_3h) AS complete_cost_rolling_3h
FROM vm_forecast_features
""").fetchdf()

display(result)

,total_rows,total_vms,available_targets,available_cost_lag_1h,complete_cost_rolling_3h
0,68852,100,63125,63123,62062


In [20]:
result = con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE target_next_hour IS NOT NULL
          AND resource_cost_index IS NOT NULL
          AND cost_lag_1h IS NOT NULL
          AND cost_lag_2h IS NOT NULL
          AND cost_rolling_3h IS NOT NULL
          AND cpu_rolling_3h IS NOT NULL
          AND memory_rolling_3h IS NOT NULL
    ) AS complete_forecast_rows,

    COUNT(*) - COUNT(DISTINCT (vm_id, hour_utc))
        AS duplicate_vm_hours

FROM vm_forecast_features
""").fetchdf()

display(result)

,total_rows,complete_forecast_rows,duplicate_vm_hours
0,68852,61629,0


In [21]:
# Create processed-data directory
output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Export records passing our core forecasting checks
output_file = output_dir / "vm_forecasting_features.parquet"

con.execute("""
COPY (
    SELECT *
    FROM vm_forecast_features
    WHERE target_next_hour IS NOT NULL
      AND resource_cost_index IS NOT NULL
      AND cost_lag_1h IS NOT NULL
      AND cost_lag_2h IS NOT NULL
      AND cost_rolling_3h IS NOT NULL
      AND cpu_rolling_3h IS NOT NULL
      AND memory_rolling_3h IS NOT NULL
    ORDER BY vm_id, hour_utc
)
TO ?
(FORMAT PARQUET)
""", [str(output_file)])

print("Forecast dataset exported:", output_file)
print("File exists:", output_file.exists())

Forecast dataset exported: E:\Cloud_cost_capstone\data\processed\vm_forecasting_features.parquet
File exists: True
